# 第 6 章 感知机（Perceptron）

## 6.1 感知机的概念

**感知机是什么？**

感知机（Perceptron）是最简单的一种**二分类模型**：

* 输入：多个信号（通常用 0 或 1 表示）；
* 输出：一个信号（0 或 1）。

可以把它想象成一个“带条件的开关”：

* 收到一堆输入信号；
* 给每个信号一个“重要程度”（权重）；
* 把加权后的结果加起来；
* 如果总和“大于某个阈值”，就输出 1（开），否则输出 0（关）。

设有两个输入 $x_1, x_2$，对应的权重为 $w_1, w_2$，阈值为 $\theta$，输出为 $y$，则感知机可以写成：

$$
y = \begin{cases}
0, & w_1 x_1 + w_2 x_2 \le \theta \
1, & w_1 x_1 + w_2 x_2 > \theta
\end{cases}
$$

* $w_1, w_2$：**权重**，控制每个输入的重要程度，权重越大，对应的输入越“有话语权”；
* $\theta$：**阈值**，控制神经元“容易被激活”的程度。

**总结**：
感知机是一个“输入加权求和 + 阈值判断”的模型，用来做最简单的二分类决策，是神经网络的基础积木块。

## 6.2 简单逻辑电路与感知机

我们可以用感知机来实现一些简单的逻辑门：与门、与非门、或门。

### 6.2.1 与门（AND gate）

**与门的规则**：
只有当两个输入都为 1 时，输出才为 1，否则为 0。

真值表：

| $x_1$ | $x_2$ | AND 输出 $y$ |
| ----- | ----- | ---------- |
| 0     | 0     | 0          |
| 0     | 1     | 0          |
| 1     | 0     | 0          |
| 1     | 1     | 1          |

用感知机来实现与门，只要找到一组参数 $(w_1, w_2, \theta)$，让上面真值表成立即可。
例如：

* $(w_1, w_2, \theta) = (0.5, 0.5, 0.7)$
* 或 $(1.0, 1.0, 1.0)$

这样的参数有无数个，只要满足“只有 (1, 1) 时加权和超过阈值”即可。

**总结**：
与门感知机用来表达“两个条件都满足才成立”的逻辑。

### 6.2.2 与非门（NAND gate）

**与非门的规则**：
与门的输出取反：

* 只有当 $x_1$ 和 $x_2$ 都为 1 时输出 0；
* 其他情况输出 1。

真值表：

| $x_1$ | $x_2$ | NAND 输出 $y$ |
| ----- | ----- | ----------- |
| 0     | 0     | 1           |
| 0     | 1     | 1           |
| 1     | 0     | 1           |
| 1     | 1     | 0           |

可以用如下参数实现 NAND 门，例如：

* $(w_1, w_2, \theta) = (-0.5, -0.5, -0.7)$

其实，只要把 AND 门的权重和阈值**符号全部取反**，就可以从 AND 变成 NAND。

**总结**：
与非门非常重要 —— 任何逻辑电路都可以用 NAND 门组合出来，所以用感知机表示 NAND 门很有价值。

### 6.2.3 或门（OR gate）

**或门的规则**：
只要有一个输入为 1，输出就是 1。

真值表：

| $x_1$ | $x_2$ | OR 输出 $y$ |
| ----- | ----- | --------- |
| 0     | 0     | 0         |
| 0     | 1     | 1         |
| 1     | 0     | 1         |
| 1     | 1     | 1         |

一个可行的参数例子：

* $(w_1, w_2, \theta) = (0.5, 0.5, 0)$

**总结**：
或门表示“只要有一个条件满足就可以”的逻辑，感知机同样可以轻松表达。

## 6.3 感知机的实现（Python）

### 6.3.1 不用 NumPy 的简单实现

先手写一个最简单的 AND 函数：

In [1]:
def AND(x1, x2):
    w1, w2, theta = 0.5, 0.5, 0.7
    res = x1 * w1 + x2 * w2
    if res <= theta:
        return 0
    else:
        return 1

print(AND(0, 0))  # 0
print(AND(1, 0))  # 0
print(AND(0, 1))  # 0
print(AND(1, 1))  # 1

0
0
0
1


* 只有 `AND(1, 1)` 输出 1，其余都输出 0，符合 AND 门的真值表。
* 这段代码直接体现了感知机的计算过程：**加权求和 → 跟阈值比较 → 输出 0 或 1**。

**总结**：
这个版本的 AND 适合用来展示“感知机的基本计算流程”。

### 6.3.2 引入权重和偏置（NumPy 版本）

为了后续扩展到更多输入，更方便地写成向量形式，我们稍微改造一下公式。

把原来的阈值 $\theta$ 换成偏置 $b$，令：

$$
b = -\theta
$$

感知机的表达式就变成：

$$
y = \begin{cases}
0, & b + w_1 x_1 + w_2 x_2 \le 0 \
1, & b + w_1 x_1 + w_2 x_2 > 0
\end{cases}
$$

* $w_1, w_2$：**权重**，控制输入的重要性；
* $b$：**偏置（bias）**，控制“神经元被激活的难易程度”。

  * $b$ 大，激活更容易（更偏向输出 1）；
  * $b$ 小（甚至负），激活更困难（更偏向输出 0）。

下面用 NumPy 写成向量形式实现 AND、NAND、OR。

#### 与门 AND

In [2]:
import numpy as np

def AND(x1, x2):
    x = np.array([x1, x2])      # 输入向量
    w = np.array([0.5, 0.5])    # 权重向量
    b = -0.7                    # 偏置（相当于 theta = 0.7）
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

print(AND(0, 0))  # 0
print(AND(1, 0))  # 0
print(AND(0, 1))  # 0
print(AND(1, 1))  # 1

0
0
0
1


#### 与非门 NAND

In [3]:
import numpy as np

def NAND(x1, x2):
    x = np.array([x1, x2])
    w = np.array([-0.5, -0.5])
    b = 0.7
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

print(NAND(0, 0))  # 1
print(NAND(1, 0))  # 1
print(NAND(0, 1))  # 1
print(NAND(1, 1))  # 0

1
1
1
0


#### 或门 OR

In [4]:
def OR(x1, x2):
    x = np.array([x1, x2])
    w = np.array([0.5, 0.5])
    b = -0.2
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

print(OR(0, 0))  # 0
print(OR(1, 0))  # 1
print(OR(0, 1))  # 1
print(OR(1, 1))  # 1

0
1
1
1


* AND / NAND / OR 这三个函数代码结构几乎一样；
* **唯一的区别只在于权重 $w$ 和偏置 $b$ 的取值不同**；
* 这说明：**改变参数，感知机就能表示不同的逻辑功能**。

**总结**：
通过参数（权重+偏置）来控制感知机的功能，是之后“通过学习自动调整参数”的基础。

## 6.4 感知机的局限：异或门问题

来看一个稍微“刁钻”的逻辑门：**异或门（XOR gate）**。

**XOR 规则**：
只有当 $x_1$ 和 $x_2$ 其中 **恰好一个为 1** 时，输出才为 1。

真值表：

| $x_1$ | $x_2$ | XOR 输出 $y$ |
| ----- | ----- | ---------- |
| 0     | 0     | 0          |
| 0     | 1     | 1          |
| 1     | 0     | 1          |
| 1     | 1     | 0          |

问题来了：
**一个单层感知机，能实现 XOR 吗？**

答案是：**不能**。

### 6.4.1 线性可分与不可分

我们把输入看成平面上的点：

* $(0, 0)$, $(0, 1)$, $(1, 0)$, $(1, 1)$

对于 OR 门，感知机决策可以写成：

$$
y = \begin{cases}
0, & -0.5 + x_1 + x_2 \le 0 \
1, & -0.5 + x_1 + x_2 > 0
\end{cases}
$$

这相当于用一条直线：

$$
-0.5 + x_1 + x_2 = 0
$$

把平面分成两块区域：一块输出 0，一块输出 1。
这种情况叫做**线性可分**：

> 可以用一条直线把两类数据“完全分开”。

但 XOR 的 0 和 1 分布是“交叉”在一起的：

* 输出为 1 的点：$(0, 1)$ 和 $(1, 0)$
* 输出为 0 的点：$(0, 0)$ 和 $(1, 1)$

你可以想象一下，怎么画一条直线，能把这两类点完全分开？
—— **做不到**。

因此，XOR 是**非线性可分**的，单层感知机只能画“直线”，自然就无能为力了。

**总结：**

* **线性空间 / 线性可分**：能被一条直线（高维中是一条“超平面”）分开的空间；
* **非线性空间 / 非线性可分**：需要曲线或更复杂边界才能分开。

> 单层感知机只能解决“用一条直线能分开”的问题（线性可分），遇到 XOR 这种“交叉分布”的数据就会失败。

## 6.5 多层感知机：用感知机实现 XOR

虽然单层感知机不能直接表示 XOR，但我们可以**组合多个感知机**来实现它。

思路是：
用已经会的 AND、NAND、OR 当作“积木”，搭一个多层结构，最终实现 XOR。

### 6.5.1 逻辑组合思路

XOR 的真值表是：

| $x_1$ | $x_2$ | XOR 输出 $y$ |
| ----- | ----- | ---------- |
| 0     | 0     | 0          |
| 0     | 1     | 1          |
| 1     | 0     | 1          |
| 1     | 1     | 0          |

一种常见的构造方式是：

1. 第 1 层：计算

   * $s_1 = \text{NAND}(x_1, x_2)$
   * $s_2 = \text{OR}(x_1, x_2)$
2. 第 2 层：计算

   * $y = \text{AND}(s_1, s_2)$

验证一下：

| $x_1$ | $x_2$ | NAND | OR | AND(NAND, OR) = XOR |
| ----- | ----- | ---- | -- | ------------------- |
| 0     | 0     | 1    | 0  | 0                   |
| 0     | 1     | 1    | 1  | 1                   |
| 1     | 0     | 1    | 1  | 1                   |
| 1     | 1     | 0    | 1  | 0                   |

刚好就是 XOR！

### 6.5.2 代码实现多层感知机 XOR

我们直接复用之前写好的 AND、NAND、OR：

In [5]:
import numpy as np

def AND(x1, x2):
    x = np.array([x1, x2])
    w = np.array([0.5, 0.5])
    b = -0.7
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

def NAND(x1, x2):
    x = np.array([x1, x2])
    w = np.array([-0.5, -0.5])
    b = 0.7
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

def OR(x1, x2):
    x = np.array([x1, x2])
    w = np.array([0.5, 0.5])
    b = -0.2
    tmp = np.sum(w * x) + b
    if tmp <= 0:
        return 0
    else:
        return 1

def XOR(x1, x2):
    s1 = NAND(x1, x2)  # 第 1 层神经元 1
    s2 = OR(x1, x2)    # 第 1 层神经元 2
    y = AND(s1, s2)    # 第 2 层神经元（输出层）
    return y

print(XOR(0, 0))  # 0
print(XOR(1, 0))  # 1
print(XOR(0, 1))  # 1
print(XOR(1, 1))  # 0

0
1
1
0


**结构理解：**

* 第 0 层：输入层，输入 $(x_1, x_2)$；
* 第 1 层：由 NAND 和 OR 两个感知机构成；
* 第 2 层：由 AND 感知机构成输出层。

这就是一个**多层感知机（Multi-layer Perceptron, MLP）** 的最简单例子。

### 6.5.3 小结：从感知机到神经网络

* 单层感知机：只能画“一条直线”做分类；
* 多层感知机：通过堆叠多层，可以组合出更复杂的边界（相当于用多条直线拼出“曲线”）；
* 异或门 XOR：是多层感知机“能做而单层做不了”的经典例子。

**总结**：
多层感知机通过“多层 + 多个感知机组合”能表示非线性空间，是现代神经网络（深度学习）的雏形和理论基础。